In [4]:
# =========================
# Llama.cpp GGUF benchmark in Colab
# Замер скорости LLM на step-runtime prompt
# =========================

!nvidia-smi


import os
import json
import time
import statistics
from pathlib import Path

import torch
from huggingface_hub import hf_hub_download
from llama_cpp import Llama

os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

REPO_ID = "bartowski/Meta-Llama-3.1-8B-Instruct-GGUF"
FILENAME = "Meta-Llama-3.1-8B-Instruct-Q4_K_M.gguf"

MODEL_PATH = hf_hub_download(
    repo_id=REPO_ID,
    filename=FILENAME,
    local_dir="/content/models"
)

print("MODEL_PATH:", MODEL_PATH)
print("Model file size GB:", round(Path(MODEL_PATH).stat().st_size / 1024**3, 2))
print("torch cuda:", torch.cuda.is_available())
print("gpu:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "no gpu")




N_CTX = 4096
N_GPU_LAYERS = -1
N_THREADS = 4
N_BATCH = 512

MAX_NEW_TOKENS = 256
REPEATS = 3
USE_LLAMA3_CHAT_TEMPLATE = True



CHAT_PERSONA = """
STYLE GUIDELINES:
- You are "inAssist", a smart calendar AI assistant.
- Language: Russian (always answer in Russian).
- Tone: Friendly, concise, professional. Avoid excessive emojis.
- Constraint: Do not use Markdown (bold/italic) in the `assistant_message` or legacy `reply_text`.
"""

RUNTIME_CONTEXT_PROMPT = """
RUNTIME CONTEXT:
- Current time: {current_time}
- User timezone: {timezone}

Use this runtime context together with the JSON payload context. If they conflict, trust the JSON payload context.
"""

DYNAMIC_INPUT_PROMPT = """
DYNAMIC INPUT:
{dynamic_input_json}
"""

STEP_SYSTEM_PROMPT = """
{persona}

TASK:
You are not a simple intent router. You are the controller of a calendar agent.
Your job is to decide exactly one NEXT STEP for the gateway.

INPUT FORMAT:
The user payload is a JSON object with:
- `text`: the newest user message or the current instruction text for this step
- `context`: time and timezone metadata
- `state`: the current session state with:
  - `messages`: human dialogue history
  - `completed_actions`: tool calls already executed by the gateway
  - `tool_observations`: real results returned by tools
  - `working_state`: current goal, status, pending action, resolved entities
  - `memory_summary`: optional short summary of older context
- optional `all_context`: raw backup history

HOW TO READ STATE:
1. `messages` = what the user and assistant said.
2. `completed_actions` = what the system already did.
3. `tool_observations` = what external tools really returned.
4. `working_state` = the current plan and unresolved entities.
5. `all_context` is only a fallback helper, not the source of truth.
6. `completed_actions` and `tool_observations` accumulate across the current task, so prefer the latest relevant observation instead of restarting the task.
7. Current time and timezone are provided in the runtime context block before the JSON payload.

IMPORTANT CONTEXT RULES:
- Prefer structured state over raw history.
- If the newest user message is short, corrective, or referential ("нет, на 10 надо", "не это", "вторую", "переименуй это"),
  resolve it using `working_state`, `completed_actions`, `tool_observations`, and `messages`.
- Do not restart the whole task from scratch if the state already contains progress.
- Do not repeat the same tool call if the required observation is already present.

AVAILABLE GATEWAY TOOLS:

1. `find_event`
Use when you need to find one or more existing events by approximate title or query text.
Arguments:
{{
  "query": "string",
  "time_min": "ISO or null",
  "time_max": "ISO or null",
  "max_results": 10
}}

2. `list_events`
Use when you need events for a whole time period, not a fuzzy title search.
Arguments:
{{
  "start": "ISO",
  "end": "ISO",
  "query": "string or null",
  "max_results": 20
}}

3. `get_free_slots`
Use when you need calendar availability in a time range.
Arguments:
{{
  "start": "ISO",
  "end": "ISO",
  "min_duration_minutes": 30
}}

4. `create_event`
Use when you already know the event details and the gateway should create it now.
Arguments:
{{
  "title": "string",
  "start_time": "ISO",
  "duration_minutes": integer,
  "location": "string or null",
  "description": "string or null"
}}

5. `update_event`
Use when you already know `event_id` and what must change.
Arguments:
{{
  "event_id": "string",
  "updates": {{
    "title": "string or null",
    "start_time": "ISO or null",
    "duration_minutes": "integer or null",
    "location": "string or null",
    "description": "string or null"
  }}
}}

6. `delete_event`
Use only when the event must truly be deleted.
Arguments:
{{
  "event_id": "string"
}}

OBSERVATION CONVENTIONS:
- The gateway gives you normalized observation objects for reasoning.
- `find_event` and `list_events` observations usually contain items with id, summary, start, end and status.
- `get_free_slots` observations usually contain slots or ranked_slots.
- `create_event` and `update_event` observations may contain event.
- `delete_event` observations may contain deleted_event_id.

DECISION MODES:

1. `tool_call`
Use when the gateway must execute one tool next.
Return exactly one tool call, not a multi-step plan.

2. `clarify`
Use when critical information is still missing even after reading the current state.
Return a short Russian clarification question to the user.
No tool call.

3. `finish`
Use when you already have enough information in state and observations to answer the user or end the task.
Return a short Russian final message.
No tool call.

WHEN TO FINISH VS CALL A TOOL:
- If the user asks general chat or task splitting, usually finish immediately.
- If the user asks to create/update/delete something and all required identifiers/details are already known, call the write tool.
- If the user asks for schedule summary and you do not have events yet, call `list_events`.
- If the user asks to find a slot and you do not have availability yet, call `get_free_slots`.
- If observations already contain enough slot/event data to answer, finish.
- If observations already contain enough data to perform the next calendar mutation, call the mutation tool.

IMPORTANT SAFETY RULES:
- Prefer `update_event` over delete+create when simple modification is enough.
- For swapping two events, usually:
  1. find the first event
  2. find the second event
  3. update one
  4. update the other
  Do not delete both events unless deletion is explicitly required.
- Do not hallucinate event IDs. Use only IDs that appear in state or observations.
- Do not invent slots or events that are not present in observations.
- Return exactly one current next step. Do not output a long executable text plan.

STATE PATCH RULES:
- `state_patch` is optional.
- Use it to update `working_state.goal`, `working_state.status`, `working_state.plan_summary`,
  `working_state.pending_action`, or `working_state.resolved_entities`.
- Keep patches small and useful.

OUTPUT FORMAT (JSON ONLY, NO EXTRA TEXT):
{{
  "response_type": "tool_call" | "clarify" | "finish",
  "assistant_message": "string or null",
  "response_payload": {{ ... optional structured data ... }},
  "next_gateway_action": {{
    "type": "none" | "tool_call",
    "tool_call": {{
      "tool_name": "find_event" | "list_events" | "get_free_slots" | "create_event" | "update_event" | "delete_event",
      "arguments": {{ ... }}
    }} or null
  }},
  "state_patch": {{ ... }}
}}
"""




STEP_REQUEST = {
    "text": "Поменяй местами на вторник визит к хирургу и игру в футбол.",
    "context": {
        "current_time": "2026-04-03T10:00:00+03:00",
        "timezone": "Europe/Moscow",
        "work_start_hour": 9,
        "work_end_hour": 18
    },
    "state": {
        "messages": [
            {
                "role": "user",
                "text": "Поменяй местами на вторник визит к хирургу и игру в футбол."
            }
        ],
        "completed_actions": [
            {
                "tool_name": "find_event",
                "arguments": {
                    "query": "хирург",
                    "time_min": "2026-04-07T00:00:00+03:00",
                    "time_max": "2026-04-07T23:59:00+03:00",
                    "max_results": 10
                }
            },
            {
                "tool_name": "find_event",
                "arguments": {
                    "query": "футбол",
                    "time_min": "2026-04-07T00:00:00+03:00",
                    "time_max": "2026-04-07T23:59:00+03:00",
                    "max_results": 10
                }
            }
        ],
        "tool_observations": [
            {
                "tool_name": "find_event",
                "result": {
                    "items": [
                        {
                            "id": "test_mini_ev_0001",
                            "summary": "визит к хирургу",
                            "start": "2026-04-07T10:00:00+03:00",
                            "end": "2026-04-07T10:30:00+03:00",
                            "status": "confirmed"
                        }
                    ]
                }
            },
            {
                "tool_name": "find_event",
                "result": {
                    "items": [
                        {
                            "id": "test_mini_ev_0002",
                            "summary": "игра в футбол",
                            "start": "2026-04-07T18:00:00+03:00",
                            "end": "2026-04-07T19:30:00+03:00",
                            "status": "confirmed"
                        }
                    ]
                }
            }
        ],
        "working_state": {
            "goal": "swap_two_events",
            "resolved_entities": {
                "first_event_id": "test_mini_ev_0001",
                "second_event_id": "test_mini_ev_0002"
            }
        },
        "memory_summary": ""
    },
    "conversation": None,
    "all_context": None
}


EXPECTED_RESPONSE = {
    "response_type": "tool_call",
    "assistant_message": None,
    "response_payload": {},
    "next_gateway_action": {
        "type": "tool_call",
        "tool_call": {
            "tool_name": "update_event",
            "arguments": {
                "event_id": "test_mini_ev_0001",
                "updates": {
                    "start_time": "2026-04-07T18:00:00+03:00",
                    "duration_minutes": 30
                }
            }
        }
    },
    "state_patch": {}
}




def build_user_payload_prompt(step_request):
    context = step_request["context"]

    runtime_context = RUNTIME_CONTEXT_PROMPT.format(
        current_time=context["current_time"],
        timezone=context["timezone"]
    ).strip()

    dynamic_input_json = json.dumps(
        step_request,
        ensure_ascii=False,
        indent=2
    )

    dynamic_input = DYNAMIC_INPUT_PROMPT.format(
        dynamic_input_json=dynamic_input_json
    ).strip()

    return runtime_context + "\n\n" + dynamic_input


def build_full_prompt(step_request):
    system_prompt = STEP_SYSTEM_PROMPT.format(
        persona=CHAT_PERSONA.strip()
    ).strip()

    user_prompt = build_user_payload_prompt(step_request)

    if USE_LLAMA3_CHAT_TEMPLATE:
        return (
            "<|begin_of_text|>"
            "<|start_header_id|>system<|end_header_id|>\n\n"
            f"{system_prompt}"
            "<|eot_id|>"
            "<|start_header_id|>user<|end_header_id|>\n\n"
            f"{user_prompt}"
            "<|eot_id|>"
            "<|start_header_id|>assistant<|end_header_id|>\n\n"
        )

    return (
        "SYSTEM:\n"
        f"{system_prompt}\n\n"
        "USER:\n"
        f"{user_prompt}\n\n"
        "ASSISTANT:\n"
    )


PROMPT = build_full_prompt(STEP_REQUEST)

print("Prompt assembled.")
print("Prompt characters:", len(PROMPT))
print("Prompt preview:")
print(PROMPT[:1200])
print("\n...")



def cuda_sync():
    if torch.cuda.is_available():
        torch.cuda.synchronize()


def token_count(llm, text, add_bos=True):
    return len(llm.tokenize(text.encode("utf-8"), add_bos=add_bos))


def benchmark_once(llm, prompt, max_new_tokens):
    prompt_tokens = llm.tokenize(prompt.encode("utf-8"), add_bos=True)

    if len(prompt_tokens) + max_new_tokens > N_CTX:
        raise ValueError(
            f"Контекст не помещается: prompt_tokens={len(prompt_tokens)}, "
            f"max_new_tokens={max_new_tokens}, n_ctx={N_CTX}. "
            f"Увеличь N_CTX или уменьши MAX_NEW_TOKENS."
        )


    llm.reset()
    cuda_sync()
    prefill_start = time.perf_counter()

    llm.eval(prompt_tokens)

    cuda_sync()
    prefill_seconds = time.perf_counter() - prefill_start

    generated_tokens = []

    cuda_sync()
    generation_start = time.perf_counter()

    for _ in range(max_new_tokens):
        token = llm.sample(
            temp=0.0,
            top_k=1,
            top_p=1.0
        )

        generated_tokens.append(token)

        if token == llm.token_eos():
            break

        llm.eval([token])

    cuda_sync()
    generation_seconds = time.perf_counter() - generation_start

    total_seconds = prefill_seconds + generation_seconds

    generated_text = llm.detokenize(generated_tokens).decode(
        "utf-8",
        errors="ignore"
    )

    return {
        "prompt_tokens": len(prompt_tokens),
        "generated_tokens": len(generated_tokens),
        "prefill_seconds": prefill_seconds,
        "generation_seconds": generation_seconds,
        "total_seconds": total_seconds,
        "prompt_tokens_per_second": len(prompt_tokens) / prefill_seconds if prefill_seconds > 0 else None,
        "generation_tokens_per_second": len(generated_tokens) / generation_seconds if generation_seconds > 0 else None,
        "total_effective_tokens_per_second": (len(prompt_tokens) + len(generated_tokens)) / total_seconds if total_seconds > 0 else None,
        "generated_text": generated_text
    }


def print_result(result, title="RESULT"):
    print(f"\n========== {title} ==========")
    print(f"Prompt tokens:                 {result['prompt_tokens']}")
    print(f"Generated tokens:              {result['generated_tokens']}")
    print(f"Prefill seconds:               {result['prefill_seconds']:.3f}")
    print(f"Generation seconds:            {result['generation_seconds']:.3f}")
    print(f"Total seconds:                 {result['total_seconds']:.3f}")
    print(f"Prompt eval tokens/sec:        {result['prompt_tokens_per_second']:.2f}")
    print(f"Generation tokens/sec:         {result['generation_tokens_per_second']:.2f}")
    print(f"Total effective tokens/sec:    {result['total_effective_tokens_per_second']:.2f}")
    print("\nGenerated text preview:")
    print(result["generated_text"][:1500])


def mean(values):
    return statistics.mean(values)


def std(values):
    return statistics.stdev(values) if len(values) > 1 else 0.0




model_path = Path(MODEL_PATH)
assert model_path.exists(), f"Файл модели не найден: {MODEL_PATH}"

print("\nLoading model...")
load_start = time.perf_counter()

llm = Llama(
    model_path=str(model_path),
    n_ctx=N_CTX,
    n_gpu_layers=N_GPU_LAYERS,
    n_threads=N_THREADS,
    n_batch=N_BATCH,
    verbose=False
)

cuda_sync()
load_seconds = time.perf_counter() - load_start

print(f"Model loaded in {load_seconds:.3f} seconds.")
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))




prompt_tokens = token_count(llm, PROMPT, add_bos=True)
print("\n========== PROMPT INFO ==========")
print(f"Prompt tokens: {prompt_tokens}")
print(f"N_CTX:         {N_CTX}")
print(f"Max new:       {MAX_NEW_TOKENS}")
print(f"Fits context:  {prompt_tokens + MAX_NEW_TOKENS <= N_CTX}")




print("\nWarmup...")
_ = benchmark_once(llm, "Привет", max_new_tokens=8)
print("Warmup done.")



results = []

for i in range(REPEATS):
    print(f"\nRun {i + 1}/{REPEATS}...")
    result = benchmark_once(
        llm=llm,
        prompt=PROMPT,
        max_new_tokens=MAX_NEW_TOKENS
    )
    results.append(result)
    print_result(result, title=f"RUN {i + 1}")


prefill_values = [r["prefill_seconds"] for r in results]
generation_values = [r["generation_seconds"] for r in results]
total_values = [r["total_seconds"] for r in results]
prompt_tps_values = [r["prompt_tokens_per_second"] for r in results]
gen_tps_values = [r["generation_tokens_per_second"] for r in results]
total_tps_values = [r["total_effective_tokens_per_second"] for r in results]
generated_tokens_values = [r["generated_tokens"] for r in results]

print("\n\n========== FINAL SUMMARY ==========")
print(f"Model path:                       {MODEL_PATH}")
print(f"Load time:                        {load_seconds:.3f} sec")
print(f"N_CTX:                            {N_CTX}")
print(f"N_GPU_LAYERS:                     {N_GPU_LAYERS}")
print(f"N_BATCH:                          {N_BATCH}")
print(f"Prompt tokens:                    {prompt_tokens}")
print(f"Max new tokens:                   {MAX_NEW_TOKENS}")
print()
print(f"Mean generated tokens:            {mean(generated_tokens_values):.1f}")
print(f"Mean prefill seconds:             {mean(prefill_values):.3f} ± {std(prefill_values):.3f}")
print(f"Mean generation seconds:          {mean(generation_values):.3f} ± {std(generation_values):.3f}")
print(f"Mean total seconds:               {mean(total_values):.3f} ± {std(total_values):.3f}")
print()
print(f"Mean prompt eval tokens/sec:      {mean(prompt_tps_values):.2f}")
print(f"Mean generation tokens/sec:       {mean(gen_tps_values):.2f}")
print(f"Mean total effective tokens/sec:  {mean(total_tps_values):.2f}")

print("\n========== EXPECTED RESPONSE FOR THIS STEP ==========")
print(json.dumps(EXPECTED_RESPONSE, ensure_ascii=False, indent=2))

print("\n========== LAST GENERATED TEXT ==========")
print(results[-1]["generated_text"])

Thu May  7 21:19:18 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   40C    P8             15W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Meta-Llama-3.1-8B-Instruct-Q4_K_M.gguf:   0%|          | 0.00/4.92G [00:00<?, ?B/s]

MODEL_PATH: /content/models/Meta-Llama-3.1-8B-Instruct-Q4_K_M.gguf
Model file size GB: 4.58
torch cuda: True
gpu: Tesla T4
Prompt assembled.
Prompt characters: 8375
Prompt preview:
<|begin_of_text|><|start_header_id|>system<|end_header_id|>

STYLE GUIDELINES:
- You are "inAssist", a smart calendar AI assistant.
- Language: Russian (always answer in Russian).
- Tone: Friendly, concise, professional. Avoid excessive emojis.
- Constraint: Do not use Markdown (bold/italic) in the `assistant_message` or legacy `reply_text`.

TASK:
You are not a simple intent router. You are the controller of a calendar agent.
Your job is to decide exactly one NEXT STEP for the gateway.

INPUT FORMAT:
The user payload is a JSON object with:
- `text`: the newest user message or the current instruction text for this step
- `context`: time and timezone metadata
- `state`: the current session state with:
  - `messages`: human dialogue history
  - `completed_actions`: tool calls already executed by the gateway
  

llama_context: n_ctx_seq (4096) < n_ctx_train (131072) -- the full capacity of the model will not be utilized


Model loaded in 16.418 seconds.
CUDA available: True
GPU: Tesla T4

========== PROMPT INFO ==========
Prompt tokens: 2166
N_CTX:         4096
Max new:       256
Fits context:  True

Warmup...
Warmup done.

Run 1/3...

========== RUN 1 ==========
Prompt tokens:                 2166
Generated tokens:              256
Prefill seconds:               2.492
Generation seconds:            7.891
Total seconds:                 10.383
Prompt eval tokens/sec:        869.35
Generation tokens/sec:         32.44
Total effective tokens/sec:    233.27

Generated text preview:
{
  "response_type": "tool_call",
  "assistant_message": null,
  "response_payload": null,
  "next_gateway_action": {
    "type": "tool_call",
    "tool_call": {
      "tool_name": "update_event",
      "arguments": {
        "event_id": "test_mini_ev_0001",
        "updates": {
          "start_time": "2026-04-07T18:00:00+03:00"
        }
      }
    }
  },
  "state_patch": {
    "working_state": {
      "status": "pending_updat